# Fake News Detection - שלב 2: בניית מודלים

**שם:** [שמך] | **ת.ז.:** [מספר זהות]

---

### מה נעשה בנוטבוק הזה:
1. טעינת הנתונים המעובדים משלב 1
2. מודלים קלאסיים: Naive Bayes, SVM, Logistic Regression
3. רשת נוירונים: LSTM
4. השוואה בין כל המודלים

## 1. התקנות וייבוא

In [ ]:
!pip install tensorflow scikit-learn matplotlib seaborn pandas numpy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn - מודלים קלאסיים
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, confusion_matrix, classification_report)
from sklearn.pipeline import Pipeline

# TensorFlow - LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, LSTM, Dense, Dropout,
                                      Bidirectional, GlobalMaxPooling1D)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. טעינת הנתונים

In [ ]:
# העלאת הקובץ ל-Colab
from google.colab import files
print('העלי את הקובץ dataset_processed.csv')
uploaded = files.upload()

In [ ]:
df = pd.read_csv('dataset_processed.csv')

# ניקוי שורות בעייתיות
df = df.dropna(subset=['title_lemma'])
df = df[df['title_lemma'].str.strip() != '']
df = df.reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'FAKE: {(df["label"]==0).sum():,} | REAL: {(df["label"]==1).sum():,}')
df.head(3)

## 3. חלוקה לTrain / Test

In [ ]:
X = df['title_lemma']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train):,}')
print(f'Test size:  {len(X_test):,}')
print(f'Train label dist: {y_train.value_counts().to_dict()}')
print(f'Test  label dist: {y_test.value_counts().to_dict()}')

---
## 4. מודלים קלאסיים

### פונקציית עזר להערכת מודל

In [ ]:
results = {}  # נשמור תוצאות של כל המודלים

def evaluate_model(name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)

    results[name] = {'Accuracy': acc, 'F1': f1, 'Precision': prec, 'Recall': rec}

    print(f'\n=== {name} ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'F1 Score:  {f1:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'\n{classification_report(y_true, y_pred, target_names=["FAKE", "REAL"])}')

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['FAKE', 'REAL'],
                yticklabels=['FAKE', 'REAL'], ax=ax)
    ax.set_title(f'Confusion Matrix - {name}', fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    plt.tight_layout()
    plt.show()

    return acc, f1

### 4.1 Naive Bayes

In [ ]:
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('clf',   MultinomialNB(alpha=0.1))
])

nb_pipeline.fit(X_train, y_train)
y_pred_nb = nb_pipeline.predict(X_test)

evaluate_model('Naive Bayes', y_test, y_pred_nb)

### 4.2 SVM (LinearSVC)

In [ ]:
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('clf',   LinearSVC(C=1.0, max_iter=2000))
])

svm_pipeline.fit(X_train, y_train)
y_pred_svm = svm_pipeline.predict(X_test)

evaluate_model('SVM', y_test, y_pred_svm)

### 4.3 Logistic Regression

In [ ]:
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('clf',   LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs'))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

evaluate_model('Logistic Regression', y_test, y_pred_lr)

---
## 5. רשת נוירונים - Bidirectional LSTM

In [ ]:
# פרמטרים
MAX_WORDS   = 15000   # גודל אוצר מילים
MAX_LEN     = 50      # אורך מקסימלי של כותרת (במילים)
EMBED_DIM   = 64      # מימד ה-Embedding
LSTM_UNITS  = 64
BATCH_SIZE  = 128
EPOCHS      = 15

# Tokenization
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'Train padded shape: {X_train_pad.shape}')
print(f'Test  padded shape: {X_test_pad.shape}')
print(f'Vocab size: {len(tokenizer.word_index):,}')

In [ ]:
# בניית המודל
def build_lstm_model():
    model = Sequential([
        Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
        Bidirectional(LSTM(LSTM_UNITS, return_sequences=True)),
        Dropout(0.3),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

lstm_model = build_lstm_model()
lstm_model.summary()

In [ ]:
# אימון
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

history = lstm_model.fit(
    X_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# גרפי אימון
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Accuracy', color='#3498db')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   color='#e74c3c')
axes[0].set_title('LSTM - Accuracy לאורך האימון', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train Loss', color='#3498db')
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='#e74c3c')
axes[1].set_title('LSTM - Loss לאורך האימון', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# הערכת LSTM
y_pred_lstm_prob = lstm_model.predict(X_test_pad, verbose=0)
y_pred_lstm = (y_pred_lstm_prob > 0.5).astype(int).flatten()

evaluate_model('Bidirectional LSTM', y_test, y_pred_lstm)

---
## 6. השוואה בין כל המודלים

In [ ]:
# טבלת השוואה
df_results = pd.DataFrame(results).T.round(4)
df_results = df_results.sort_values('F1', ascending=False)
df_results.index.name = 'Model'

print('=== השוואת ביצועים ===')
df_results

In [ ]:
# ויזואליזציה - השוואת מדדים
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df_results))
width = 0.2
metrics = ['Accuracy', 'F1', 'Precision', 'Recall']
colors  = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, df_results[metric], width, label=metric,
           color=color, edgecolor='black', alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df_results.index, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('השוואת ביצועי המודלים - Fake News Detection', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='80% threshold')
ax.grid(axis='y', alpha=0.3)

# הוספת ערכים על הבארים
for bar in ax.patches:
    if bar.get_height() > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 7. ניתוח שגיאות - מה המודל הטוב ביותר טועה?

In [ ]:
# נבדוק שגיאות של המודל הטוב ביותר (LSTM)
best_model_name = df_results.index[0]
print(f'המודל הטוב ביותר: {best_model_name}')

# נבחר predictions של המודל הטוב
pred_map = {
    'Naive Bayes'          : y_pred_nb,
    'SVM'                  : y_pred_svm,
    'Logistic Regression'  : y_pred_lr,
    'Bidirectional LSTM'   : y_pred_lstm
}
best_preds = pred_map[best_model_name]

# שגיאות
X_test_series = X_test.reset_index(drop=True)
y_test_series = y_test.reset_index(drop=True)

errors_df = pd.DataFrame({
    'title'     : X_test_series,
    'actual'    : y_test_series,
    'predicted' : best_preds
})
errors_df = errors_df[errors_df['actual'] != errors_df['predicted']]
errors_df['actual']    = errors_df['actual'].map({0: 'FAKE', 1: 'REAL'})
errors_df['predicted'] = errors_df['predicted'].map({0: 'FAKE', 1: 'REAL'})

print(f'\nסה"כ שגיאות: {len(errors_df):,}')
print(f'False Positives (FAKE predicted as REAL): {len(errors_df[errors_df["actual"]=="FAKE"]):,}')
print(f'False Negatives (REAL predicted as FAKE): {len(errors_df[errors_df["actual"]=="REAL"]):,}')
print(f'\n10 דוגמאות לשגיאות:')
errors_df.head(10)

## 8. שמירת המודל הטוב ביותר

In [ ]:
import pickle

# שמירת LSTM
lstm_model.save('lstm_model.keras')

# שמירת Tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# שמירת SVM / LR pipeline (לשימוש מהיר)
with open('svm_pipeline.pkl', 'wb') as f:
    pickle.dump(svm_pipeline, f)

with open('lr_pipeline.pkl', 'wb') as f:
    pickle.dump(lr_pipeline, f)

print('Models saved!')

# הורדת הקבצים מ-Colab למחשב
files.download('lstm_model.keras')
files.download('tokenizer.pkl')
files.download('svm_pipeline.pkl')
files.download('lr_pipeline.pkl')

---
## סיכום תוצאות

In [ ]:
print('=' * 55)
print('        MODEL COMPARISON SUMMARY')
print('=' * 55)
print(df_results.to_string())
print('=' * 55)
print(f'\nהמודל הטוב ביותר לפי F1: {df_results["F1"].idxmax()}')
print(f'F1 Score: {df_results["F1"].max():.4f}')